In [1]:
print("Kernel working")

Kernel working


In [2]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [3]:
# Step 1: Load the dataset

import os

raw_path = r"C:\Users\akula\OneDrive\Desktop\DOCUMENTS\Seanergy_Preparation\Projects\Automated_Email_Classification\data\raw"

os.listdir(raw_path)

['spam.csv']

In [4]:
df = pd.read_csv("../data/raw/spam.csv", encoding="latin-1")

In [5]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [6]:
df.shape

(5572, 5)

In [7]:
df.columns

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='str')

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   v1          5572 non-null   str  
 1   v2          5572 non-null   str  
 2   Unnamed: 2  50 non-null     str  
 3   Unnamed: 3  12 non-null     str  
 4   Unnamed: 4  6 non-null      str  
dtypes: str(5)
memory usage: 217.8 KB


In [9]:
# Remove unnecessary columns

df = df[['v1', 'v2']]

df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [10]:
# Rename columns

df = df.rename(columns={
    'v1': 'target',
    'v2': 'message'
})

df.head()

,target,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [11]:
# Check class distribution

df['target'].value_counts()

target
ham     4825
spam     747
Name: count, dtype: int64

In [12]:
# Check missing values

df.isnull().sum()

target     0
message    0
dtype: int64

In [13]:
# Check duplicate messages

df.duplicated().sum()

np.int64(403)

In [14]:
# Remove duplicate messages

df = df.drop_duplicates(keep='first')

# Reset the index
df = df.reset_index(drop=True)

df.shape

(5169, 2)

In [15]:
# Check the cleaned dataset

df.head()

,target,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [16]:
# Import libraries for text preprocessing

import nltk
import string
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

In [17]:
# Download required NLTK resources

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\akula\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [18]:
# Initialize Porter Stemmer

ps = PorterStemmer()

In [19]:
# Function for text preprocessing

def transform_text(text):
    # Convert text to lowercase
    text = text.lower()
    
    # Tokenize the text
    text = nltk.word_tokenize(text)
    
    # Keep only alphanumeric words
    text = [word for word in text if word.isalnum()]
    
    # Remove stopwords and punctuation
    stop_words = set(stopwords.words('english'))
    text = [word for word in text if word not in stop_words]
    
    # Apply stemming
    text = [ps.stem(word) for word in text]
    
    # Join the words back into a single string
    return " ".join(text)

In [20]:
# Download tokenizer resource

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\akula\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [22]:
# Download the tokenizer data required by the current NLTK version

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\akula\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [23]:
# Test text preprocessing on one message

transform_text(df['message'].iloc[0])

'go jurong point crazi avail bugi n great world la e buffet cine got amor wat'

In [24]:
# Apply text preprocessing to all messages

df['transformed_text'] = df['message'].apply(transform_text)

df.head()

,target,message,transformed_text
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,ham,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [25]:
# Check for empty transformed messages

(df['transformed_text'].str.strip() == '').sum()

np.int64(9)

In [26]:
# View messages that became empty after preprocessing

df[df['transformed_text'].str.strip() == ''][['target', 'message']]

,target,message
248,ham,What you doing?how are you?
939,ham,Where @
2675,ham,Can a not?
2935,ham,staff.science.nus.edu.sg/~phyhcmk/teaching/pc1323
3191,ham,:)
4019,ham,G.W.R
4276,ham,:( but your not here....
4500,ham,:-) :-)
4896,ham,\ER


In [27]:
# Keep the original message when preprocessing produces an empty result

empty_mask = df['transformed_text'].str.strip() == ''

df.loc[empty_mask, 'transformed_text'] = (
    df.loc[empty_mask, 'message']
    .str.lower()
    .str.strip()
)

# Check again
(df['transformed_text'].str.strip() == '').sum()

np.int64(0)

In [28]:
# Final preprocessing check

print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (5169, 3)

Missing values:
target              0
message             0
transformed_text    0
dtype: int64

Duplicate rows: 0

Columns:
['target', 'message', 'transformed_text']


In [30]:
# Save the processed dataset

processed_path = r"C:\Users\akula\OneDrive\Desktop\DOCUMENTS\Seanergy_Preparation\Projects\Automated_Email_Classification\data\processed\spam_processed.csv"

df.to_csv(processed_path, index=False)

print("Processed dataset saved successfully!")
print(processed_path)

Processed dataset saved successfully!
C:\Users\akula\OneDrive\Desktop\DOCUMENTS\Seanergy_Preparation\Projects\Automated_Email_Classification\data\processed\spam_processed.csv
